# Week 4 - Pandas Fundamentals

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 4 - Pandas Fundamentals**

---

## Session topics (from the course outline)

- Series and DataFrames
- Importing data (CSV, Excel)
- Basic data exploration: `head()`, `info()`, `describe()`

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 2** | Identify suitable tools and techniques for analyzing different types of datasets |
| **CLO 4** | Use visualization tools to create charts, graphs and plots for effective data presentation |
| **CLO 5** | Perform exploratory data analysis (EDA) to summarize and interpret datasets |

## Reading

| Topic | Source |
|---|---|
| Series, DataFrame, Index objects | McKinney §5.1 |
| Indexing, selection, filtering | McKinney §5.2 |
| Arithmetic and data alignment | McKinney §5.2 |
| Reading and writing text format | McKinney §6.1 |
| Reading Microsoft Excel files | McKinney §6.2 |
| DataFrames | Pfaffelhuber §7.1 |
| Loading data from files | Pfaffelhuber §7.8 |

## What pandas adds to NumPy

Everything you learned in Week 3 still applies: broadcasting, boolean indexing, the `axis`
parameter. pandas adds one thing to the array, and the whole library follows from it.

**The index.** A `Series` is a NumPy array with a label for every element. A `DataFrame` is a
set of `Series` sharing one index. Because the labels travel with the data, pandas can align
two objects by label before combining them which is the behaviour that makes it useful and
the behaviour that surprises people most.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [2]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
%pip install matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

  Obtaining dependency information for matplotlib from https://files.pythonhosted.org/packages/04/93/0d239bde12b308262265c2d98909b2f0ecad2b5deeae96241642e822e9c7/matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata
  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Obtaining dependency information for contourpy>=1.0.1 from https://files.pythonhosted.org/packages/50/a9/dab08786bb4d77ef9a046b3c1be923be4e73c5d07d91f5a54e8ea9b09e41/contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Obtaining dependency information for cycler>=0.10 from https://files.pythonhosted.org/packages/e7/05/c19819d5e3d95294a6f5947fb9b9629efb316b96de511b418c53d245aae6/cycler-0.12.1-py3-none-any.whl.metadata
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Obtaining dependency information for fonttools>=4.28.2 from https://files.pythonhosted.org/packages/c1/3a/625a6dd0173e88dbea1826405b4bcbfa06c6ca0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


ModuleNotFoundError: No module named 'seaborn'

In [7]:
import pandas as pd

df = pd.read_csv("call_activity.csv")

df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110.0,45.07,197.4,99.0,16.78,244.7,91.0,11.01,10.0,3,2.70,1.0,False
1,OH,107,415,No,Yes,26,161.6,123.0,27.47,195.5,103.0,16.62,254.4,103.0,11.45,13.7,3,3.70,1.0,False
2,NJ,137,415,No,No,0,243.4,114.0,41.38,121.2,110.0,10.30,162.6,104.0,7.32,12.2,5,3.29,0.0,False
3,OH,84,408,Yes,No,0,299.4,71.0,50.90,61.9,88.0,5.26,196.9,89.0,8.86,6.6,7,1.78,2.0,False
4,OK,75,415,Yes,No,0,166.7,113.0,28.34,148.3,122.0,12.61,186.9,121.0,8.41,10.1,3,2.73,3.0,False


In [1]:
import pandas as pd

# 1. Load the CSV file
df = pd.read_csv("call_activity.csv")

# 2. Select the first 4 rows and convert them to HTML
df_to_html = df.head(4).to_html(index=False)

# 3. Save the HTML code to a file
with open("call_data.html", "w", encoding="utf-8") as file:
    file.write(df_to_html)

print("HTML file created successfully!")

HTML file created successfully!


In [ ]:
for index, row in df.iterrows():
    print(f"Row {index}: {row.to_dict()}")

---

# Part 1 - Theory and concepts

## 1.1 The `Series`

A `Series` is a one-dimensional array with an associated **index**:

```python
s = pd.Series([78.5, 91.0, 66.25], index=["Sophea", "Dara", "Chanthou"])
```

It has two components you can access separately:

| Attribute | What it is |
|---|---|
| `.values` | The underlying NumPy array |
| `.index` | The labels, as an `Index` object |

If no index is given, pandas supplies a `RangeIndex`: 0, 1, 2, … That default is why a
`Series` can look like a list while behaving differently.

## 1.2 The `DataFrame`

A `DataFrame` is a table: an ordered collection of `Series` that **share one index**. Each
column has its own dtype; the index is common to all of them.

```python
df = pd.DataFrame({
    "score": [78.5, 91.0],
    "passed": [True, True],
})
```

| Attribute | What it is |
|---|---|
| `.index` | The row labels |
| `.columns` | The column labels |
| `.dtypes` | One dtype per column |
| `.shape` | `(n_rows, n_columns)` |
| `.values` | A single 2-D NumPy array - note that mixing dtypes forces a common one, usually `object` |

The last row is worth remembering: `.values` on a mixed-type DataFrame gives up the type
information. It is rarely what you want.

## 1.3 Automatic alignment

This is the defining behaviour of pandas, and it has no equivalent in NumPy.

When two `Series` are combined, pandas matches them **by index label**, not by position. The
result carries the **union** of the two indexes, and any label present in only one of them
gives `NaN`.

```python
a = pd.Series([1, 2, 3], index=["a", "b", "c"])
b = pd.Series([10, 20, 30], index=["b", "c", "d"])
a + b
```

| label | a | b | a + b |
|---|---|---|---|
| a | 1 | — | `NaN` |
| b | 2 | 10 | 12 |
| c | 3 | 20 | 23 |
| d | — | 30 | `NaN` |

With NumPy arrays the same addition would pair element 0 with element 0 and give
`[11, 22, 33]`.

Alignment is the feature that prevents a whole class of silent errors: joining last month's
sales to this month's by row position, when the two files list branches in a different order,
produces wrong numbers with no warning. pandas refuses to do that.

The cost is that `NaN` appears where you may not have expected it. When a computation
produces unexpected `NaN`, mismatched indexes are the first thing to check.

## 1.4 Selecting data

pandas offers three ways to select, and confusing them is the most common source of errors
in the first weeks.

| Form | Selects by | Example | Notes |
|---|---|---|---|
| `df[...]` | Column name, or a boolean mask for rows | `df["score"]`, `df[df["score"] > 80]` | Convenient, and inconsistent: a string means a column, a mask means rows |
| `df.loc[rows, cols]` | **Label** | `df.loc["Sophea", "score"]` | End of a slice is **inclusive** |
| `df.iloc[rows, cols]` | **Integer position** | `df.iloc[0, 1]` | End of a slice is **exclusive**, as in Python |

Two differences between `.loc` and `.iloc` cause most of the trouble:

**Slice endpoints.** `df.loc["a":"c"]` includes `"c"`. `df.iloc[0:3]` excludes position 3.
Label slicing is inclusive because there is no "one past the end" label to name.

**What an integer means.** If the index happens to be integers, `df.loc[2]` means the row
*labelled* 2 and `df.iloc[2]` means the row in *position* 2. These are different rows unless
the index is a clean `RangeIndex`, and after any filtering it is not.

**Use `.loc` and `.iloc` explicitly** rather than relying on `df[...]`. The convenience form
is ambiguous, and the ambiguity is what produces `SettingWithCopyWarning` in Week 5.

## 1.5 Reading files

```python
pd.read_csv("data.csv")
pd.read_excel("data.xlsx", sheet_name="Sheet1")
```

`read_csv` has a large number of parameters. Six of them matter for most real files:

| Parameter | Purpose |
|---|---|
| `sep` | Field separator. `","` by default; `"\t"` for TSV, `";"` in many European exports |
| `na_values` | Extra strings to treat as missing, e.g. `["N/A", "-", "missing"]` |
| `dtype` | Force a column's type, e.g. `{"student_id": str}` to preserve leading zeros |
| `parse_dates` | Column names to convert to timestamps |
| `index_col` | Column to use as the index |
| `encoding` | `"utf-8"` normally; some Windows exports need `"latin-1"` or `"utf-8-sig"` |

Two of these prevent specific, common data losses:

- **`dtype={"id": str}`.** An identifier like `"007"` read as a number becomes `7`, and the
  leading zeros are gone permanently. Any column of codes should be read as a string.
- **`na_values`.** A column containing `"N/A"` is read as text, so the whole column becomes
  strings and arithmetic on it fails. Declaring the missing marker keeps the column numeric.

`read_excel` requires the `openpyxl` package for `.xlsx` files, which is why Notebook 00
installed it.

## 1.6 First inspection

Four methods, run in this order, on every dataset you meet:

| Method | Answers |
|---|---|
| `.shape` | How much data is there? |
| `.head()` | What does a row look like? |
| `.info()` | What types are the columns, and how many values are missing? |
| `.describe()` | What is the range and centre of each numeric column? |

For non-numeric columns, `.describe(include="object")` reports count, number of distinct
values, most frequent value and its frequency. `.value_counts()` gives the full distribution
of one column, and `normalize=True` turns the counts into proportions.

`.info()` is the most informative of the four, because it answers the type question and the
missing-data question together and those are the two questions that determine what you can
do next.

---

---

# Part 2 - Code example

Complete and executable. The order is deliberate: the `Series` and its index first, then
alignment, then the DataFrame, then selection, then files.

## 2.1 The Series and its index

In [2]:
scores = pd.Series(
    [78.5, 91.0, 66.25, 84.0, 72.5],
    index=["Sophea", "Dara", "Chanthou", "Rithy", "Bopha"],
    name="final_score",
)

print(scores)

print("\n.values ->", type(scores.values).__name__, scores.values)
print(".index  ->", list(scores.index))
print(".dtype  ->", scores.dtype)
print(".name   ->", scores.name)

# Selection works by label, and by position.
print("\nBy label    scores['Dara']  :", scores["Dara"])
print("By position scores.iloc[1]   :", scores.iloc[1])

# Everything from Week 3 still works: boolean masks, vectorised arithmetic.
print("\nAbove 75:")
print(scores[scores > 75])
print("\nScaled by 1.1 (vectorised, as in NumPy):")
print((scores * 1.1).round(2))

Sophea      78.50
Dara        91.00
Chanthou    66.25
Rithy       84.00
Bopha       72.50
Name: final_score, dtype: float64

.values -> ndarray [78.5  91.   66.25 84.   72.5 ]
.index  -> ['Sophea', 'Dara', 'Chanthou', 'Rithy', 'Bopha']
.dtype  -> float64
.name   -> final_score

By label    scores['Dara']  : 91.0
By position scores.iloc[1]   : 91.0

Above 75:
Sophea    78.5
Dara      91.0
Rithy     84.0
Name: final_score, dtype: float64

Scaled by 1.1 (vectorised, as in NumPy):
Sophea       86.35
Dara        100.10
Chanthou     72.88
Rithy        92.40
Bopha        79.75
Name: final_score, dtype: float64


## 2.2 Alignment

Two `Series` whose indexes differ. Watch what the addition does.

In [ ]:
semester_1 = pd.Series([70, 82, 65, 90],
                       index=["Sophea", "Dara", "Chanthou", "Rithy"])
semester_2 = pd.Series([75, 88, 71, 68],
                       index=["Dara", "Chanthou", "Rithy", "Mealea"])

print("Semester 1:")
print(semester_1)
print("\nSemester 2:")
print(semester_2)

print("\nsemester_1 + semester_2:")
print(semester_1 + semester_2)

In [ ]:
# What happened: pandas matched the labels, not the positions.
print("Index of semester_1 :", list(semester_1.index))
print("Index of semester_2 :", list(semester_2.index))
print("Union of the two    :", sorted(set(semester_1.index) | set(semester_2.index)))
print("\nSophea is only in semester 1 and Mealea only in semester 2,")
print("so both give NaN. The three students in both are added.")

# NumPy would have paired position with position, silently.
print("\nThe same addition as NumPy arrays:")
print(semester_1.to_numpy() + semester_2.to_numpy())
print("\nThat pairs Sophea with Dara. It is the wrong answer, computed without")
print("complaint. Alignment is what prevents it.")

# To treat a missing label as zero rather than NaN, say so explicitly.
print("\nWith fill_value=0:")
print(semester_1.add(semester_2, fill_value=0))

## 2.3 The DataFrame

A table of `Series` sharing one index.

In [ ]:
def generate_enrolment_frame(n: int = 120, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Build a reproducible enrolment DataFrame.

    Parameters
    ----------
    n : int
        Number of students.
    seed : int
        Seed for reproducibility.

    Returns
    -------
    pandas.DataFrame
        One row per student.
    """
    generator = np.random.default_rng(seed)

    faculties = ["Engineering", "Business", "Science", "Arts"]
    faculty = generator.choice(faculties, size=n, p=[0.35, 0.30, 0.20, 0.15])

    credits = generator.choice([12, 15, 18], size=n, p=[0.25, 0.50, 0.25])
    ability = generator.normal(0, 1, size=n)

    frame = pd.DataFrame({
        "student_id": [f"S{2026000 + i}" for i in range(n)],
        "faculty": faculty,
        "year_level": generator.choice([1, 2, 3, 4], size=n, p=[0.30, 0.30, 0.25, 0.15]),
        "credits": credits,
        "gpa": np.clip(2.6 + ability * 0.5, 0, 4).round(2),
        "attendance_pct": np.clip(78 + ability * 9 + generator.normal(0, 5, n), 0, 100).round(1),
        "scholarship": generator.choice([True, False], size=n, p=[0.22, 0.78]),
    })
    frame["fee_usd"] = (frame["credits"] * 95.0).round(2)
    return frame


students = generate_enrolment_frame()

print("shape  :", students.shape)
print("columns:", list(students.columns))
print("index  :", students.index)

print("\n", students.head())

In [ ]:
# A DataFrame is columns of Series sharing an index.
gpa_column = students["gpa"]

print("One column is a Series:", type(gpa_column).__name__)
print("Its index is the DataFrame's index:", gpa_column.index.equals(students.index))

print("\nEach column has its own dtype:")
print(students.dtypes)

# .values collapses to a single array, which forces a common dtype.
print("\nstudents.values dtype:", students.values.dtype)
print("Mixing text, numbers and booleans forces 'object', losing the type")
print("information. Use .to_numpy() on a single numeric column instead:")
print("  students['gpa'].to_numpy().dtype ->", students["gpa"].to_numpy().dtype)

## 2.4 Selection: `[]`, `.loc` and `.iloc`

The distinction that causes the most confusion in the first weeks.

In [ ]:
# Give the frame a meaningful index so that label and position differ.
indexed = students.set_index("student_id")
print(indexed.head(3))

print("\n--- by LABEL with .loc ---")
print("indexed.loc['S2026002', 'gpa'] :", indexed.loc["S2026002", "gpa"])
print("\nindexed.loc['S2026000':'S2026002', ['faculty', 'gpa']]:")
print(indexed.loc["S2026000":"S2026002", ["faculty", "gpa"]])

print("\n--- by POSITION with .iloc ---")
print("indexed.iloc[2, 3] :", indexed.iloc[2, 3])
print("\nindexed.iloc[0:3, [0, 3]]:")
print(indexed.iloc[0:3, [0, 3]])

In [ ]:
# The endpoint rule: .loc is inclusive, .iloc is exclusive.
label_slice = indexed.loc["S2026000":"S2026002"]
position_slice = indexed.iloc[0:3]

print(f".loc['S2026000':'S2026002'] returned {len(label_slice)} rows (endpoint INCLUDED)")
print(f".iloc[0:3]                  returned {len(position_slice)} rows (endpoint EXCLUDED)")

print("\nLabel slicing is inclusive because there is no 'one past the end'")
print("label to name. Position slicing follows the usual Python rule.")

In [ ]:
# Where the distinction stops being academic: an integer index after filtering.
high_gpa = students[students["gpa"] > 3.0]

print("After filtering, the index keeps the ORIGINAL row labels:")
print(high_gpa.index[:8].tolist())

print("\n.iloc[0] -> the FIRST row of the filtered frame:")
print(high_gpa.iloc[0][["student_id", "gpa"]].to_dict())

first_label = high_gpa.index[0]
print(f"\n.loc[{first_label}] -> the row LABELLED {first_label}, which is the same row here:")
print(high_gpa.loc[first_label][["student_id", "gpa"]].to_dict())

print(f"\nBut .loc[0] asks for the row labelled 0, and 0 is"
      f" {'still' if 0 in high_gpa.index else 'no longer'} in this index.")
try:
    high_gpa.loc[0]
except KeyError:
    print("  -> KeyError. Row 0 did not survive the filter.")
    print("\nThis is why the two must not be confused: after any filtering,")
    print("label and position have parted company.")

In [ ]:
# Boolean selection, and combining conditions.
engineering = students[students["faculty"] == "Engineering"]
print(f"Engineering students: {len(engineering)}")

# & | ~ with parentheses, exactly as in NumPy.
target = students[(students["gpa"] > 3.0) & (students["attendance_pct"] > 85)]
print(f"GPA above 3.0 AND attendance above 85%: {len(target)}")

# .isin() replaces a chain of == comparisons.
stem = students[students["faculty"].isin(["Engineering", "Science"])]
print(f"Engineering or Science: {len(stem)}")

# .loc takes a mask and a column list together, which is the clearest form.
print("\nTop five by GPA, selected columns:")
print(
    students.loc[students["gpa"] > 3.4, ["student_id", "faculty", "gpa", "attendance_pct"]]
    .sort_values("gpa", ascending=False)
    .head()
)

## 2.5 Reading files

The cell below writes a deliberately awkward CSV, then reads it twice: once naively, once
with the parameters that handle it. The file is created inside the notebook so that nothing
needs downloading.

In [ ]:
CSV_PATH = Path("week04_sample.csv")

MESSY_CSV = """student_id,faculty,enrolled_on,credits,fee_usd,note
0042,Engineering,2026-03-02,15,1425.00,
0043,Business,2026-03-03,12,1140.00,late enrolment
0044,Science,2026-03-03,N/A,1710.00,
0045,Arts,2026-03-04,15,N/A,deferred
0046,Engineering,2026-03-05,18,1710.00,
"""


def write_sample_csv(path: Path = CSV_PATH) -> Path:
    """Write the sample CSV used by this notebook, and return its path."""
    path.write_text(MESSY_CSV, encoding="utf-8")
    return path


write_sample_csv()
print("Wrote", CSV_PATH.resolve().name)
print("\nRaw contents:")
print(MESSY_CSV)

In [12]:
# Read 1: the default. This CSV does not contain a 'student_id' column.
naive = pd.read_csv("call_activity.csv")

print("Default read_csv:")
print(naive)
print("\nDtypes:")
print(naive.dtypes)

print("\nThis file uses a different schema than the sample:")
print(f"  first row state code: {naive.loc[0, 'State']!r}")
print("  there is no 'student_id' column in this dataset")
print("  the columns are loaded with their default pandas dtypes for this file")

Default read_csv:
     State  Account length  Area code International plan Voice mail plan  \
0       KS             128        415                 No             Yes   
1       OH             107        415                 No             Yes   
2       NJ             137        415                 No              No   
3       OH              84        408                Yes              No   
4       OK              75        415                Yes              No   
...    ...             ...        ...                ...             ...   
3328    AZ             192        415                 No             Yes   
3329    WV              68        415                 No              No   
3330    RI              28        510                 No              No   
3331    CT             184        510                Yes              No   
3332    TN              74        415                 No             Yes   

      Number vmail messages  Total day minutes  Total day calls  \
0 

In [14]:
# Read 2: the same file, read properly.
clean = pd.read_csv(
    "call_activity.csv",
    dtype={"student_id": str},      # preserve the leading zeros
    na_values=["N/A", ""],          # treat these as missing
    parse_dates=["enrolled_on"],    # convert to timestamps
)

print("With the right parameters:")
print(clean)
print("\nDtypes:")
print(clean.dtypes)

print("\nstudent_id preserved      :", repr(clean.loc[0, "student_id"]))
print("credits is numeric         :", pd.api.types.is_numeric_dtype(clean["credits"]))
print("enrolled_on is a timestamp :", pd.api.types.is_datetime64_any_dtype(clean["enrolled_on"]))
print("missing values found       :", int(clean.isna().sum().sum()))

# Date arithmetic is now possible, which it was not before.
span = clean["enrolled_on"].max() - clean["enrolled_on"].min()
print("enrolment window           :", span.days, "days")

ValueError: Missing column provided to 'parse_dates': 'enrolled_on'

In [ ]:
# Writing files. Excel needs openpyxl, installed in Notebook 00.
clean.to_csv("week04_clean.csv", index=False)
print("Wrote week04_clean.csv")

try:
    clean.to_excel("week04_clean.xlsx", index=False)
    reread = pd.read_excel("week04_clean.xlsx")
    print("Wrote and re-read week04_clean.xlsx, shape:", reread.shape)
    Path("week04_clean.xlsx").unlink()
except ImportError:
    print("openpyxl is not installed; run: pip install openpyxl")

Path("week04_clean.csv").unlink()

# index=False matters: without it, the index is written as an unnamed first
# column, and reading the file back produces a spurious 'Unnamed: 0'.
clean.to_csv("week04_withindex.csv")
print("\nWritten WITHOUT index=False, then read back:")
print(pd.read_csv("week04_withindex.csv").columns.tolist())
Path("week04_withindex.csv").unlink()

## 2.6 First inspection

Four methods, in this order, on every dataset you meet.

In [ ]:
print("=" * 70)
print("1. .shape — how much data is there?")
print("=" * 70)
print(f"{students.shape[0]} rows x {students.shape[1]} columns")

print("\n" + "=" * 70)
print("2. .head() — what does a row look like?")
print("=" * 70)
print(students.head())

print("\n" + "=" * 70)
print("3. .info() — types and missing values together")
print("=" * 70)
students.info()

In [ ]:
print("=" * 70)
print("4. .describe() — range and centre of the numeric columns")
print("=" * 70)
print(students.describe().round(2))

print("\n" + "=" * 70)
print(".describe(include='object') — the non-numeric columns")
print("=" * 70)
print(students.describe(include="object"))

print("\nNote that 'year_level' appears in the numeric summary. It is stored as")
print("an integer, but it is ordinal (Week 1), so its mean is not meaningful.")
print("describe() cannot know that. You can.")

In [ ]:
# value_counts: the full distribution of one column.
print("Counts:")
print(students["faculty"].value_counts())

print("\nProportions:")
print((students["faculty"].value_counts(normalize=True) * 100).round(1))

print("\nCross-tabulation of two columns:")
print(pd.crosstab(students["faculty"], students["scholarship"]))

# Sorting, which .describe() does not give you.
print("\nFive highest GPAs:")
print(students.nlargest(5, "gpa")[["student_id", "faculty", "gpa"]])

In [ ]:
# Closing the loop: a first look at the frame, visually.
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

axes[0].hist(students["gpa"], bins=18, color=PALETTE[0], edgecolor="white")
axes[0].axvline(students["gpa"].mean(), color=PALETTE[3], linewidth=2,
                label=f"mean {students['gpa'].mean():.2f}")
axes[0].set_title("(a) GPA distribution")
axes[0].set_xlabel("GPA")
axes[0].set_ylabel("students")
axes[0].legend(fontsize=8)

counts = students["faculty"].value_counts().sort_values()
axes[1].barh(counts.index, counts.to_numpy(), color=PALETTE[1], edgecolor="white")
axes[1].set_title("(b) Students by faculty")
axes[1].set_xlabel("students")
axes[1].grid(axis="y", alpha=0)

axes[2].scatter(students["attendance_pct"], students["gpa"], s=35,
                alpha=0.6, color=PALETTE[2], edgecolor="white", linewidth=0.5)
axes[2].set_title("(c) Attendance against GPA")
axes[2].set_xlabel("attendance (%)")
axes[2].set_ylabel("GPA")

plt.tight_layout()
plt.show()

print(f"Correlation, attendance and GPA: r = "
      f"{students['attendance_pct'].corr(students['gpa']):.3f}")

---

# Part 3 - Guided workshop (in class)

Four tasks. Each has a verification cell.

### Task 1 - Build a Series and read its index

**Problem.** Build `fees`, a `Series` of the fee due for each faculty, indexed by faculty
name:

| faculty | fee |
|---|---|
| Engineering | 1710.0 |
| Business | 1425.0 |
| Science | 1620.0 |
| Arts | 1140.0 |

Then compute `fees_with_levy`: the same fees with a 5% levy added.

**Hints**

- `pd.Series(values, index=labels)`.
- A vectorised multiply applies to every element, as in NumPy.

**Expected output.** Both Series printed, followed by `Task 1 complete`.

In [ ]:
# TODO 1a: build the Series, indexed by faculty name.
fees = None  # replace this line

# TODO 1b: add a 5% levy to every fee.
fees_with_levy = None  # replace this line

print(fees)
print("\nWith 5% levy:")
print(fees_with_levy)

In [ ]:
# Verification for Task 1.
expected_index = ["Engineering", "Business", "Science", "Arts"]
expected_values = [1710.0, 1425.0, 1620.0, 1140.0]

if fees is None or fees_with_levy is None:
    print("Not complete: fill in both TODO 1a and TODO 1b.")
elif not isinstance(fees, pd.Series):
    print(f"Expected a pandas Series, got {type(fees).__name__}.")
elif sorted(fees.index) != sorted(expected_index):
    print("The index should hold the four faculty names. Got:", list(fees.index))
elif not np.allclose(fees.reindex(expected_index).to_numpy(), expected_values):
    print("The values do not match the table.")
elif not np.allclose(fees_with_levy.reindex(expected_index).to_numpy(),
                     np.array(expected_values) * 1.05):
    print("fees_with_levy should be fees * 1.05.")
else:
    print("Task 1 complete.")

### Task 2 - Watch alignment happen

**Problem.** Two `Series` hold last year's and this year's enrolment counts, for overlapping
but different sets of faculties. Compute `change`, the difference (`this_year - last_year`).

Then answer, in `only_this_year`, which faculty labels appear in `this_year` but not in
`last_year`.

**Hints**

- Subtracting two `Series` aligns them by label automatically. Do not convert to arrays.
- A label present in only one Series gives `NaN` in the result.
- `this_year.index.difference(last_year.index)` gives the labels in one and not the other.

**Expected output.** A `change` Series containing two `NaN` values, followed by
`Task 2 complete`.

In [ ]:
last_year = pd.Series({"Engineering": 40, "Business": 35, "Science": 22, "Arts": 18})
this_year = pd.Series({"Business": 38, "Science": 25, "Arts": 15, "Medicine": 12})

# TODO 2a: compute this_year minus last_year, letting pandas align the labels.
change = None  # replace this line

# TODO 2b: which labels are in this_year but not last_year? Return an Index or a list.
only_this_year = None  # replace this line

print(change)
print("\nOnly in this year:", list(only_this_year) if only_this_year is not None else None)

In [ ]:
# Verification for Task 2.
if change is None or only_this_year is None:
    print("Not complete: fill in both TODO 2a and TODO 2b.")
elif not isinstance(change, pd.Series):
    print(f"`change` should be a Series, got {type(change).__name__}.")
    print("If you used .to_numpy(), you lost the alignment that this task is about.")
else:
    expected = this_year - last_year
    if not change.index.sort_values().equals(expected.index.sort_values()):
        print("The index should be the union of both:", sorted(expected.index))
    elif not np.allclose(change.sort_index().to_numpy(),
                         expected.sort_index().to_numpy(), equal_nan=True):
        print("The values do not match this_year - last_year.")
    elif sorted(only_this_year) != ["Medicine"]:
        print("only_this_year should contain just 'Medicine'. Got:", list(only_this_year))
    else:
        n_nan = int(change.isna().sum())
        print(f"Task 2 complete. {n_nan} labels appear in only one Series, giving NaN.")

### Task 3 - Select with `.loc` and `.iloc`

**Problem.** Using the `students` DataFrame, produce three selections.

1. `first_three_gpa` — the `gpa` of the first three rows, **by position**.
2. `row_five_faculty` — the `faculty` of the row at position 5, **by position**.
3. `high_attendance` — a DataFrame of `student_id`, `faculty` and `gpa` for every student
   whose `attendance_pct` is above 90, selected with `.loc`.

**Hints**

- `.iloc[rows, cols]` takes integers; columns can be given by position too.
- For (3), `.loc` accepts a boolean mask and a list of column names in one call:
  `df.loc[mask, ["a", "b"]]`.

**Expected output.** Three results of the right shapes, followed by `Task 3 complete`.

In [ ]:
# TODO 3a: the gpa of the first three rows, by position.
first_three_gpa = None

# TODO 3b: the faculty of the row at position 5, by position.
row_five_faculty = None

# TODO 3c: student_id, faculty and gpa where attendance_pct > 90, using .loc
high_attendance = None

print("First three GPAs   :", list(first_three_gpa) if first_three_gpa is not None else None)
print("Faculty at row 5   :", row_five_faculty)
print("\nAttendance above 90%:")
print(high_attendance.head() if high_attendance is not None else None)

In [ ]:
# Verification for Task 3.
problems = []

if first_three_gpa is None:
    problems.append("first_three_gpa is still None")
elif len(first_three_gpa) != 3:
    problems.append(f"first_three_gpa should have 3 values, has {len(first_three_gpa)}")
elif not np.allclose(np.asarray(first_three_gpa, dtype=float),
                     students["gpa"].iloc[:3].to_numpy()):
    problems.append("first_three_gpa values do not match the first three rows")

if row_five_faculty is None:
    problems.append("row_five_faculty is still None")
elif row_five_faculty != students["faculty"].iloc[5]:
    problems.append(f"row_five_faculty should be {students['faculty'].iloc[5]!r}")

if high_attendance is None:
    problems.append("high_attendance is still None")
elif not isinstance(high_attendance, pd.DataFrame):
    problems.append("high_attendance should be a DataFrame")
elif list(high_attendance.columns) != ["student_id", "faculty", "gpa"]:
    problems.append(f"high_attendance columns should be "
                    f"['student_id', 'faculty', 'gpa'], got {list(high_attendance.columns)}")
elif len(high_attendance) != int((students["attendance_pct"] > 90).sum()):
    problems.append("high_attendance has the wrong number of rows; check the threshold")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print(f"Task 3 complete. {len(high_attendance)} students have attendance above 90%.")

### Task 4 - Read a file properly

**Problem.** Read `week04_sample.csv` a third time, into `careful`, so that:

- `student_id` keeps its leading zeros,
- `N/A` and empty fields are treated as missing,
- `enrolled_on` is parsed as a date,
- `student_id` becomes the **index**.

**Hints**

- All four are `read_csv` parameters: `dtype`, `na_values`, `parse_dates`, `index_col`.
- `index_col` accepts a column name.

**Expected output.** A DataFrame indexed by student ID with four columns, followed by
`Task 4 complete`.

In [ ]:
# TODO 4: read the file with the four parameters described above.
careful = None  # replace this line

print(careful)
print("\nIndex :", list(careful.index) if careful is not None else None)
print("Dtypes:")
print(careful.dtypes if careful is not None else None)

In [ ]:
# Verification for Task 4.
if careful is None:
    print("Not complete: `careful` is still None.")
elif not isinstance(careful, pd.DataFrame):
    print(f"Expected a DataFrame, got {type(careful).__name__}.")
else:
    problems = []
    if careful.index.name != "student_id":
        problems.append(f"the index should be student_id, it is {careful.index.name!r}")
    elif list(careful.index)[0] != "0042":
        problems.append(f"leading zeros were lost: first index value is "
                        f"{list(careful.index)[0]!r}, expected '0042'")
    if "credits" in careful.columns and not pd.api.types.is_numeric_dtype(careful["credits"]):
        problems.append("credits is not numeric; declare the missing marker in na_values")
    if "enrolled_on" in careful.columns and not pd.api.types.is_datetime64_any_dtype(
            careful["enrolled_on"]):
        problems.append("enrolled_on was not parsed as a date")
    if int(careful.isna().sum().sum()) < 4:
        problems.append("fewer missing values than expected; check na_values")

    if problems:
        print("Not complete:")
        for p in problems:
            print("  -", p)
    else:
        print("Task 4 complete. All four parameters applied correctly.")

---

# Part 4 - Homework

Run **Restart Kernel and Run All Cells** before submitting, then push to your
`puc-data-analysis` repository.

## Problem 1 - A column report

Week 1's `profile_dataset` described a whole frame. This one describes every column, and
returns the result as a `DataFrame` so it can be sorted and filtered like any other table.

**Problem.** Write `column_report(df)`.

**Requirements.** Return a `DataFrame` with **one row per column of the input**, indexed by
column name, with the index named `"column"`, and exactly these columns in this order:

| Column | Type | Value |
|---|---|---|
| `dtype` | `str` | The column's dtype, as a string |
| `n_missing` | `int` | Count of missing values |
| `pct_missing` | `float` | Percentage missing, `0`–`100`, rounded to 2 decimal places |
| `n_unique` | `int` | Number of distinct non-missing values |
| `is_numeric` | `bool` | The result of `pd.api.types.is_numeric_dtype(series)` |
| `minimum` | `float` | Smallest value for numeric columns, `NaN` otherwise |
| `maximum` | `float` | Largest value for numeric columns, `NaN` otherwise |
| `mean` | `float` | Mean for numeric columns, `NaN` otherwise |

The rows must appear in the same order as the columns of the input.

> **A warning that is not a trick question.** In Week 1 you used
> `df.select_dtypes(include="number")`, which does **not** select boolean columns. Here you
> are asked for `pd.api.types.is_numeric_dtype`, which **does** count a boolean column as
> numeric.
>
> These two pandas functions disagree about booleans. Neither is wrong: `select_dtypes` is
> about storage categories, while `is_numeric_dtype` is about whether arithmetic is defined —
> and it is, because `True` is 1. The mean of a boolean column is the proportion of `True`
> values, which is a genuinely useful number.
>
> The lesson is that "is this column numeric?" has more than one reasonable answer, so the
> specification has to say which one it wants. Read the spec rather than assuming
> consistency.

**Hints**

- `pd.api.types.is_numeric_dtype(series)` answers the numeric question **as this spec defines
  it**. Do not substitute `select_dtypes` or a comparison against dtype strings.
- `series.nunique()` excludes missing values by default, which is what is wanted.
- Build a list of dictionaries, then `pd.DataFrame(rows).set_index("column")`. Setting the
  index from a column named `"column"` names the index for you.
- `np.nan` for the statistics of a non-numeric column.

In [ ]:
# Problem 1 — starter code.

def column_report(df: pd.DataFrame) -> pd.DataFrame:
    """Describe every column of a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        The frame to describe.

    Returns
    -------
    pandas.DataFrame
        One row per input column, indexed by column name.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete column_report()")


report = column_report(students)
print(report)

print("\nColumns with any missing data:")
print(report[report["n_missing"] > 0])

In [ ]:
# Problem 1 — validation. All assertions must pass.

test = pd.DataFrame({
    "num":  [1.0, 2.0, 3.0, np.nan],
    "text": ["a", "b", "b", None],
    "flag": [True, False, True, True],
})

rep = column_report(test)

assert isinstance(rep, pd.DataFrame), "column_report must return a DataFrame"
assert list(rep.index) == ["num", "text", "flag"], (
    f"Rows must be one per input column, in order; got {list(rep.index)}"
)
assert rep.index.name == "column", f"The index must be named 'column'; got {rep.index.name!r}"

expected_cols = ["dtype", "n_missing", "pct_missing", "n_unique",
                 "is_numeric", "minimum", "maximum", "mean"]
assert list(rep.columns) == expected_cols, (
    f"Columns must be exactly {expected_cols} in that order; got {list(rep.columns)}"
)

# --- missing values ---
assert rep.loc["num", "n_missing"] == 1
assert rep.loc["text", "n_missing"] == 1
assert rep.loc["flag", "n_missing"] == 0
assert np.isclose(rep.loc["num", "pct_missing"], 25.0), "1 of 4 is 25.0, not 0.25"
assert np.isclose(rep.loc["flag", "pct_missing"], 0.0)

# --- distinct values exclude missing ---
assert rep.loc["num", "n_unique"] == 3, "nunique() should not count NaN"
assert rep.loc["text", "n_unique"] == 2, "'a' and 'b'; None is not counted"

# --- numeric detection, as is_numeric_dtype defines it ---
assert bool(rep.loc["num", "is_numeric"]) is True
assert bool(rep.loc["text", "is_numeric"]) is False
assert bool(rep.loc["flag", "is_numeric"]) is True, (
    "pd.api.types.is_numeric_dtype() counts a boolean column as numeric. "
    "Note this DISAGREES with select_dtypes(include='number') from Week 1 — "
    "see the warning in the problem statement."
)

# --- statistics, present for numeric columns and NaN otherwise ---
assert np.isclose(rep.loc["num", "minimum"], 1.0)
assert np.isclose(rep.loc["num", "maximum"], 3.0)
assert np.isclose(rep.loc["num", "mean"], 2.0)

# The boolean column is numeric here, so it gets real statistics.
# False is 0 and True is 1, so the mean is the proportion of True values.
assert np.isclose(rep.loc["flag", "minimum"], 0.0)
assert np.isclose(rep.loc["flag", "maximum"], 1.0)
assert np.isclose(rep.loc["flag", "mean"], 0.75), (
    "3 of the 4 values are True, so the mean is 0.75 — the proportion of True"
)

# Only the genuinely non-numeric column gets NaN.
for statistic in ["minimum", "maximum", "mean"]:
    assert pd.isna(rep.loc["text", statistic]), (
        f"{statistic} must be NaN for the text column"
    )

# --- dtype reported as a string ---
assert isinstance(rep.loc["num", "dtype"], str), "dtype must be a string"

# --- works on the notebook's own frame ---
full = column_report(students)
assert len(full) == students.shape[1], "One row per column of the input"
assert list(full.index) == list(students.columns), "Row order must match column order"
assert full["n_missing"].sum() == 0, "The students frame has no missing values"

# --- a frame with no numeric columns at all ---
text_only = column_report(pd.DataFrame({"a": ["x", "y"], "b": ["p", "q"]}))
assert not text_only["is_numeric"].any()
assert text_only["minimum"].isna().all()

print("Problem 1: all assertions passed.")

---

## Problem 2 - Load a file the way a real one arrives

**Problem.** Write `load_enrolments(path)`, which reads a CSV written by an administrative
system and returns a usable DataFrame.

The file has these problems, all of which appear in real exports:

- Identifier codes with leading zeros, which must be preserved as text.
- Missing values written as `"N/A"`, `"-"` or left blank.
- A date column stored as text.
- Column names with inconsistent capitalisation and surrounding spaces.

**Requirements.** Return a `DataFrame` in which:

| Requirement | Detail |
|---|---|
| Column names | Lower case, stripped of surrounding whitespace, spaces replaced with `_` |
| `student_id` | dtype `object`/`str`, leading zeros intact |
| `credits`, `fee_usd` | numeric dtype |
| `enrolled_on` | datetime dtype |
| Missing markers | `"N/A"`, `"-"` and empty fields all become `NaN` |
| Index | The default `RangeIndex` — do **not** set an index |

Do not drop any rows.

**Hints**

- Read first, rename afterwards. The raw names are awkward, so do not try to pass them to
  `dtype=` before you have cleaned them.
- `df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]`.
- `parse_dates` needs the name as it appears **in the file**. Reading first and converting
  with `pd.to_datetime` afterwards avoids that problem entirely.
- `pd.to_numeric(series, errors="coerce")` turns anything unconvertible into `NaN`.

In [ ]:
# Problem 2 — starter code.

RAW_CSV = """ Student ID ,Faculty,ENROLLED ON,Credits,Fee USD
0042,Engineering,2026-03-02,15,1425.00
0043,Business,2026-03-03,12,1140.00
0044,Science,2026-03-03,N/A,1710.00
0045,Arts,2026-03-04,15,-
0046,Engineering,2026-03-05,18,1710.00
0047,Business,,12,1140.00
"""

RAW_PATH = Path("week04_homework.csv")
RAW_PATH.write_text(RAW_CSV, encoding="utf-8")


def load_enrolments(path) -> pd.DataFrame:
    """Read an administrative enrolment export into a usable DataFrame.

    Parameters
    ----------
    path : str or pathlib.Path
        Location of the CSV file.

    Returns
    -------
    pandas.DataFrame
        Cleaned column names, correct dtypes, missing markers converted.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete load_enrolments()")


loaded = load_enrolments(RAW_PATH)
print(loaded)
print("\nDtypes:")
print(loaded.dtypes)
print("\nMissing per column:")
print(loaded.isna().sum())

In [ ]:
# Problem 2 — validation. All assertions must pass.
loaded = load_enrolments(RAW_PATH)

assert isinstance(loaded, pd.DataFrame), "load_enrolments must return a DataFrame"
assert len(loaded) == 6, f"No rows should be dropped; expected 6, got {len(loaded)}"

# --- column names cleaned ---
assert list(loaded.columns) == ["student_id", "faculty", "enrolled_on",
                                "credits", "fee_usd"], (
    f"Column names should be lower case, stripped and underscored; "
    f"got {list(loaded.columns)}"
)

# --- the index was left alone ---
assert isinstance(loaded.index, pd.RangeIndex), (
    "Do not set an index; the default RangeIndex is required"
)

# --- student_id keeps its leading zeros ---
assert not pd.api.types.is_numeric_dtype(loaded["student_id"]), (
    "student_id must stay text, or the leading zeros are lost"
)
assert loaded.loc[0, "student_id"] == "0042", (
    f"Expected '0042', got {loaded.loc[0, 'student_id']!r}"
)

# --- numeric columns are numeric ---
assert pd.api.types.is_numeric_dtype(loaded["credits"]), "credits must be numeric"
assert pd.api.types.is_numeric_dtype(loaded["fee_usd"]), "fee_usd must be numeric"

# --- the date column is a datetime ---
assert pd.api.types.is_datetime64_any_dtype(loaded["enrolled_on"]), (
    "enrolled_on must be a datetime dtype"
)

# --- all three missing markers were converted ---
assert pd.isna(loaded.loc[2, "credits"]), "'N/A' in credits must become NaN"
assert pd.isna(loaded.loc[3, "fee_usd"]), "'-' in fee_usd must become NaN"
assert pd.isna(loaded.loc[5, "enrolled_on"]), "An empty date field must become NaN"
assert int(loaded.isna().sum().sum()) == 3, (
    f"Exactly 3 missing values are expected; got {int(loaded.isna().sum().sum())}"
)

# --- the values that were present are correct ---
assert loaded.loc[0, "credits"] == 15
assert np.isclose(loaded.loc[0, "fee_usd"], 1425.00)
assert loaded.loc[0, "enrolled_on"] == pd.Timestamp("2026-03-02")
assert loaded["faculty"].tolist() == ["Engineering", "Business", "Science",
                                       "Arts", "Engineering", "Business"]

# --- date arithmetic works, which was the point of parsing ---
span = loaded["enrolled_on"].max() - loaded["enrolled_on"].min()
assert span.days == 3, f"Enrolment window should be 3 days; got {span.days}"

# --- it must work when called again, and on a Path or a string ---
again = load_enrolments(str(RAW_PATH))
assert again.shape == loaded.shape, "The function must accept a string path too"

print("Problem 2: all assertions passed.")

In [ ]:
# Tidy up the files this notebook created.
for name in ["week04_sample.csv", "week04_homework.csv"]:
    path = Path(name)
    if path.exists():
        path.unlink()
        print("removed", name)

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 5

Week 5 is data cleaning: missing values, duplicates, type conversion and transformation. The
`NaN` values that alignment produced today, and the ones `load_enrolments` created from
`"N/A"`, are the subject of the whole session.

Before then, read McKinney §7.1–7.2 and Pfaffelhuber §7.7.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §5.1–5.2, §6.1–6.2.

Pfaffelhuber, P. *Python for Data Analysis* (course notes). §7.1, §7.8.
Available at `https://pfaffelh.github.io/python_for_data/`.